# Week 5 Saturday Project: SG Market Dashboard
**Goal:** Combine everything from this week into a market analysis dashboard.

You'll build a `MarketDashboard` class that:
1. Loads SGX stock data with pandas (Days 1-3)
2. Computes market metrics — sector analysis, top picks, value screens (Days 2-3)
3. Gets AI commentary from Claude with structured output (Day 4)
4. Generates an HTML report you can open in a browser (Week 4 skills)

This is your first **end-to-end data + AI project** — the same pattern you'll
use for every portfolio project going forward.

---
## Part 1: Setup & Load Data

Set up imports, API key, and load the cleaned SGX data from Day 3.

In [2]:
# TODO: Setup — imports and API key
#
import pandas as pd
import anthropic
import json
import os

os.environ["ANTHROPIC_API_KEY"] = "your-key-here"   # REMOVE before pushing to GitHub!"

print("Ready!")

Ready!


In [3]:
# TODO: Load the cleaned data and explore it
#
df = pd.read_csv("data/sgx_cleaned.csv")
print(f"Loaded {len(df)} stocks")
print(f"Columns: {list(df.columns)}")
print(f"Sectors: {df['sector'].nunique()} unique")
df.head()

Loaded 20 stocks
Columns: ['ticker', 'name', 'sector', 'market_cap_m', 'price', 'dividend_yield', 'pe_ratio']
Sectors: 11 unique


,ticker,name,sector,market_cap_m,price,dividend_yield,pe_ratio
0,D05,DBS Group,Banking,108500,41.20,5.1,10.2
1,O39,OCBC Bank,Banking,65200,15.80,5.4,9.8
2,U11,UOB,Banking,55800,34.50,4.8,10.5
3,Z74,Singtel,Telecom,42100,3.25,4.9,18.3
4,C6L,Singapore Airlines,Aviation,18900,6.85,0.0,15.7


---
## Part 2: Market Metrics (pandas)

Compute the numbers that will go into your dashboard. All pandas — no AI yet.

In [4]:
# TODO: Market overview stats
#
total_market_cap = df["market_cap_m"].sum()
avg_yield = df["dividend_yield"].mean()
avg_pe = df["pe_ratio"].mean()
num_stocks = len(df)

print("=== SGX Market Overview ===")
print(f"Total stocks: {num_stocks}")
print(f"Total market cap: ${total_market_cap:,.0f}M")
print(f"Average dividend yield: {avg_yield:.1f}%")
print(f"Average PE ratio: {avg_pe:.1f}")

=== SGX Market Overview ===
Total stocks: 20
Total market cap: $509,400M
Average dividend yield: 4.2%
Average PE ratio: 15.7


In [5]:
# TODO: Sector breakdown — count, avg yield, avg PE, total market cap per sector
#
sector_stats = df.groupby("sector").agg(
  stocks=("name", "count"),
  total_cap=("market_cap_m", "sum"),
  avg_yield=("dividend_yield", "mean"),
  avg_pe=("pe_ratio", "mean")
).sort_values("total_cap", ascending=False)

print("\n=== Sector Breakdown ===")
print(sector_stats)


=== Sector Breakdown ===
                   stocks  total_cap  avg_yield     avg_pe
sector                                                    
Banking                 3     229500   5.100000  10.166667
Technology              2      49800   0.000000  29.700000
Telecom                 1      42100   4.900000  18.300000
Conglomerate            2      41300   3.700000  13.150000
Real Estate             2      29000   4.250000  14.950000
REIT - Industrial       3      24100   5.833333  16.166667
REIT - Commercial       2      23500   5.800000  14.550000
Industrial              2      22700   3.950000  15.850000
Agriculture             1      21300   4.200000  11.800000
Aviation                1      18900   0.000000  15.700000
REIT - Logistics        1       7200   6.500000  13.200000


In [6]:
# TODO: Top 5 dividend stocks and top 5 value stocks (lowest PE)
#
top_yield = df.sort_values("dividend_yield", ascending=False).head(5)
top_value = df[df["pe_ratio"] > 0].sort_values("pe_ratio").head(5)

print("=== Top 5 Dividend Stocks ===")
print(top_yield[["name", "sector", "dividend_yield", "price"]].to_string(index=False))

print("\n=== Top 5 Value Stocks (Lowest PE) ===")
print(top_value[["name", "sector", "pe_ratio", "price"]].to_string(index=False))

=== Top 5 Dividend Stocks ===
                               name            sector  dividend_yield  price
          Mapletree Logistics Trust  REIT - Logistics             6.5   1.48
Mapletree Pan Asia Commercial Trust REIT - Commercial             6.2   1.35
            Frasers Logistics Trust REIT - Industrial             6.1   1.22
         Mapletree Industrial Trust REIT - Industrial             5.8   2.18
           CapitaLand Ascendas REIT REIT - Industrial             5.6   2.78

=== Top 5 Value Stocks (Lowest PE) ===
            name      sector  pe_ratio  price
   Hongkong Land Real Estate       7.5   5.30
YZJ Shipbuilding  Industrial       8.9   2.15
       OCBC Bank     Banking       9.8  15.80
       DBS Group     Banking      10.2  41.20
             UOB     Banking      10.5  34.50


---
## Part 3: AI Commentary (Claude)

Send the pandas data to Claude and get structured analysis back.
Uses the `parse_json_response` and `validate_analysis` functions from Day 4.

In [7]:
# TODO: Bring in your Day 4 helper functions
#
def parse_json_response(raw_text):
  cleaned = raw_text.strip()
  cleaned = cleaned.removeprefix("```json").removeprefix("```")
  cleaned = cleaned.removesuffix("```")
  cleaned = cleaned.strip()
  return json.loads(cleaned)

def validate_analysis(data, required_keys):
  missing = [key for key in required_keys if key not in data]
  if missing:
      return (False, f"Missing keys: {missing}")
  return (True, data)

print("Helper functions ready!")

Helper functions ready!


In [8]:
# TODO: Get AI market commentary with structured output
#
client = anthropic.Anthropic()
MODEL = "claude-sonnet-5"

# Build a data summary string to send to Claude
data_summary = f"""SGX Market Data ({num_stocks} stocks):

Total market cap: ${total_market_cap:,.0f}M
Average dividend yield: {avg_yield:.1f}%
Average PE ratio: {avg_pe:.1f}

Sector breakdown:
{sector_stats.to_string()}

Top 5 dividend stocks:
{top_yield[['name','sector','dividend_yield','price']].to_string()}

Top 5 value stocks (lowest PE):
{top_value[['name','sector','pe_ratio','price']].to_string()}
"""

message = client.messages.create(
  model=MODEL,
  max_tokens=800,
  system="""You are a Singapore equity market analyst. Respond in valid JSON only.
Return exactly these keys:
- market_outlook: one sentence overall market assessment
- top_sector: which sector looks strongest and why (one sentence)
- dividend_pick: your top dividend stock pick and why (one sentence)
- value_pick: your top value stock pick and why (one sentence)
- risk_warning: one key risk to watch (one sentence)
No other text, just the JSON object.""",
  messages=[{"role": "user", "content": f"Analyze this SGX market data and provide your commentary:\n\n{data_summary}"}]
)

raw = message.content[0].text
commentary = parse_json_response(raw)

REQUIRED_KEYS = ["market_outlook", "top_sector", "dividend_pick", "value_pick", "risk_warning"]
valid, result = validate_analysis(commentary, REQUIRED_KEYS)

if valid:
  print("AI Commentary received and validated!")
  for key, value in commentary.items():
      print(f"  {key}: {value}")
else:
  print(f"Validation failed: {result}")

AI Commentary received and validated!
  market_outlook: SGX presents a balanced market with reasonable valuations (15.7x PE) and attractive income yields (4.2%), supported by a solid banking sector anchor.
  top_sector: Banking looks strongest with low PE ratios (~10.2x), high dividend yields (5.1%), and a substantial $229.5B market cap providing stability and value.
  dividend_pick: Mapletree Logistics Trust offers the highest yield at 6.5% among top stocks, providing strong income potential within the resilient logistics REIT space.
  value_pick: Hongkong Land stands out with the lowest PE ratio at 7.5x, suggesting significant undervaluation relative to its real estate assets.


---
## Part 4: Generate HTML Dashboard

Turn everything into a styled HTML report — data tables + AI insights.

In [9]:
# TODO: Build the HTML report
#
def build_dashboard(df, sector_stats, top_yield, top_value, commentary):
  # Build sector rows
  sector_rows = ""
  for sector, row in sector_stats.iterrows():
      sector_rows += f"""<tr>
          <td>{sector}</td>
          <td>{row['stocks']}</td>
          <td>${row['total_cap']:,.0f}M</td>
          <td>{row['avg_yield']:.1f}%</td>
          <td>{row['avg_pe']:.1f}</td>
      </tr>"""

  # Build top dividend rows
  dividend_rows = ""
  for _, row in top_yield.iterrows():
      dividend_rows += f"""<tr>
          <td>{row['name']}</td>
          <td>{row['sector']}</td>
          <td>{row['dividend_yield']}%</td>
          <td>${row['price']}</td>
      </tr>"""

  # Build top value rows
  value_rows = ""
  for _, row in top_value.iterrows():
      value_rows += f"""<tr>
          <td>{row['name']}</td>
          <td>{row['sector']}</td>
          <td>{row['pe_ratio']}</td>
          <td>${row['price']}</td>
      </tr>"""

  html = f"""<!DOCTYPE html>
<html>
<head>
  <title>SGX Market Dashboard</title>
  <style>
      body {{ font-family: 'Segoe UI', Arial, sans-serif; max-width: 900px; margin: 40px auto; padding: 0 20px; background: #f5f7fa; color: #1a1a2e; }}
      h1 {{ color: #0d47a1; border-bottom: 3px solid #0d47a1; padding-bottom: 10px; }}
      h2 {{ color: #1565c0; margin-top: 30px; }}
      .overview {{ display: grid; grid-template-columns: repeat(4, 1fr); gap: 15px; margin: 20px 0; }}
      .stat-card {{ background: white; padding: 20px; border-radius: 8px; text-align: center; box-shadow: 0 2px 4px rgba(0,0,0,0.1); }}
      .stat-card .number {{ font-size: 1.8em; font-weight: bold; color: #0d47a1; }}
      .stat-card .label {{ color: #666; font-size: 0.9em; margin-top: 5px; }}
      table {{ width: 100%; border-collapse: collapse; background: white; border-radius: 8px; overflow: hidden; box-shadow: 0 2px 4px rgba(0,0,0,0.1); margin: 15px 0; }}
      th {{ background: #0d47a1; color: white; padding: 12px; text-align: left; }}
      td {{ padding: 10px 12px; border-bottom: 1px solid #eee; }}
      tr:hover {{ background: #f0f4ff; }}
      .ai-section {{ background: white; padding: 25px; border-radius: 8px; border-left: 4px solid #4caf50; box-shadow: 0 2px 4px rgba(0,0,0,0.1); margin: 20px 0; }}
      .ai-section h3 {{ color: #4caf50; margin-top: 0; }}
      .ai-item {{ margin: 12px 0; padding: 8px 0; border-bottom: 1px solid #f0f0f0; }}
      .ai-label {{ font-weight: bold; color: #333; }}
      .footer {{ text-align: center; color: #999; margin-top: 40px; font-size: 0.85em; }}
  </style>
</head>
<body>
  <h1>SGX Market Dashboard</h1>

  <div class="overview">
      <div class="stat-card">
          <div class="number">{len(df)}</div>
          <div class="label">Stocks Tracked</div>
      </div>
      <div class="stat-card">
          <div class="number">${df['market_cap_m'].sum():,.0f}M</div>
          <div class="label">Total Market Cap</div>
      </div>
      <div class="stat-card">
          <div class="number">{df['dividend_yield'].mean():.1f}%</div>
          <div class="label">Avg Dividend Yield</div>
      </div>
      <div class="stat-card">
          <div class="number">{df['pe_ratio'].mean():.1f}</div>
          <div class="label">Avg PE Ratio</div>
      </div>
  </div>

  <h2>Sector Breakdown</h2>
  <table>
      <tr><th>Sector</th><th>Stocks</th><th>Market Cap</th><th>Avg Yield</th><th>Avg PE</th></tr>
      {sector_rows}
  </table>

  <h2>Top 5 Dividend Stocks</h2>
  <table>
      <tr><th>Name</th><th>Sector</th><th>Dividend Yield</th><th>Price</th></tr>
      {dividend_rows}
  </table>

  <h2>Top 5 Value Stocks (Lowest PE)</h2>
  <table>
      <tr><th>Name</th><th>Sector</th><th>PE Ratio</th><th>Price</th></tr>
      {value_rows}
  </table>

  <div class="ai-section">
      <h3>AI Market Commentary</h3>
      <div class="ai-item"><span class="ai-label">Market Outlook:</span> {commentary['market_outlook']}</div>
      <div class="ai-item"><span class="ai-label">Top Sector:</span> {commentary['top_sector']}</div>
      <div class="ai-item"><span class="ai-label">Dividend Pick:</span> {commentary['dividend_pick']}</div>
      <div class="ai-item"><span class="ai-label">Value Pick:</span> {commentary['value_pick']}</div>
      <div class="ai-item"><span class="ai-label">Risk Warning:</span> {commentary['risk_warning']}</div>
  </div>

  <div class="footer">Generated with pandas + Claude AI | Week 5 Saturday Project</div>
</body>
</html>"""
  return html

html = build_dashboard(df, sector_stats, top_yield, top_value, commentary)
print(f"Dashboard HTML generated! ({len(html)} characters)")

Dashboard HTML generated! (6908 characters)


---
## Part 5: Save & Open

Save the HTML file and open it in your browser.

In [10]:
# TODO: Save and open the dashboard
#
output_path = "sgx_dashboard.html"
with open(output_path, "w") as f:
  f.write(html)
print(f"Dashboard saved to {output_path}")

# Open in browser
import webbrowser
webbrowser.open(output_path)
print("Opened in browser!")

Dashboard saved to sgx_dashboard.html
Opened in browser!
